<a href="https://colab.research.google.com/github/satyam-thakur/LLM-Assisted-Container-Security-Analysis/blob/code/vul_analysis/src/notebook/quick_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔒 Container Security CVE Analysis

**LLM-Assisted Vulnerability Analysis for Container Security**

This notebook provides a complete workflow for:
- Running CVE vulnerability analysis using `runner.py`
- Optimizing the analysis agent with DSPy MIPROv2 using `optimize_miprov2.py`

---

In [ ]:
# Check GPU Status
import torch

if torch.cuda.is_available():
    print(f"🚀 GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"   CUDA Version: {torch.version.cuda}")
else:
    print("💻 Running on CPU")
    print("   Tip: Enable GPU in Colab (Runtime > Change runtime type > GPU)")

# Installation

In [3]:
# Clone repository and setup workspace
import os
import sys
from getpass import getpass

# Repository configuration
GITHUB_REPO = "https://github.com/satyam-thakur/LLM-Assisted-Container-Security-Analysis.git"
repo_root = '/content/LLM-Assisted-Container-Security-Analysis'

if not os.path.exists(repo_root):
    print("📥 Cloning repository...")
    # Check if GitHub token is available for private repos (optional)
    github_token = os.environ.get('GITHUB_TOKEN') or getpass('Enter GitHub token (or press Enter for public access): ')
    
    if github_token:
        os.environ['GITHUB_TOKEN'] = github_token
        cloning_url = GITHUB_REPO.replace("https://github.com", f"https://{github_token}@github.com")
        print("Using provided token for cloning...")
    else:
        cloning_url = GITHUB_REPO
        print("Cloning as public repository...")

    !git clone -b code {cloning_url}
    
    if os.path.exists(repo_root):
        print('Repository cloned successfully!')
    else:
        raise RuntimeError("Failed to clone repository. Please check your token and repo access.")
else:
    print('Repository already exists')

# Set up paths
os.chdir(repo_root)
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print(f'Working directory: {os.getcwd()}')
print(f'Python path updated')

📥 Cloning repository...
Cloning as public repository...


Cloning into 'LLM-Assisted-Container-Security-Analysis'...
fatal: User cancelled dialog.
bash: line 1: /dev/tty: No such device or address
error: failed to execute prompt script (exit code 1)
fatal: could not read Username for 'https://github.com': No such file or directory


RuntimeError: Failed to clone repository. Please check your token and repo access.

In [ ]:
# Install Dependencies
!pip install -q -r {REPO_DIR}/vul_analysis/src/requirements.txt
print("✅ Dependencies installed")

# Directory Setup

In [ ]:
# Setup Python Path and Working Directory
import sys

os.chdir(REPO_DIR)
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print(f"📂 Working directory: {os.getcwd()}")

# Ensure log directories exist
os.makedirs("vul_analysis/logs", exist_ok=True)
os.makedirs("vul_analysis/.cache/dspy_programs", exist_ok=True)

# Configuration

In [ ]:
# Available Models (from runner.py)
# LLM Models: Context Window | Price (In/Out per 1M)
MODEL_SPECS = {
    "gpt-5.1-pro":        "400K ctx, $1.25/$10.00, Deep vulnerability reasoning & research-grade analysis",
    "gpt-4.1":            "128K ctx, $2.00/$8.00, Reliable CVE triage & exploit explanation",
    "gpt-4.1-mini":       "128K ctx, $0.40/$1.60, Balanced baseline for academic experiments",
    "claude-3.7-sonnet":  "200K+ ctx, $3.00/$15.00, Precise patch diffing & secure-code reasoning",
    "gemini-2.5-pro":     "1M ctx, $1.25/$10.00, Large log, SBOM & supply-chain analysis",
    "deepseek-reasoner":  "128K ctx, $0.55/$2.19, Strong reasoning for CVE root-cause analysis",
    "deepseek-chat":      "128K ctx, $0.27/$1.10, Cost-effective large-scale vulnerability scans",
    "llama-4-maverick":   "1M ctx, Free (Open Weights), Private data & reproducible research",
    "qwen-qwq-32b":       "128K ctx, $1.20, Structured security reasoning & benchmarks",
    "mistral-large-3":    "256K ctx, $0.50/$1.50, Fast analysis of Dockerfiles & IaC",
    "gpt-oss-120b":       "128K ctx, Free (Infra), Open & reproducible academic findings",
    "gemini-2.0-flash":   "1M ctx, $0.10/$0.40, Cheap baseline for large experiment sweeps",
}

MODEL_LIST = list(MODEL_SPECS.keys())

print("=" * 85)
print("🤖 Available Models for CVE Analysis")
print("=" * 85)
print(f"{'Model Name':<20} | {'Specifications'}")
print("-" * 85)
for model, spec in MODEL_SPECS.items():
    print(f"{model:<20} | {spec}")
print("=" * 85)

In [ ]:
# Model Selection (Interactive)
def get_user_model_input():
    """Get and validate user model input with proper formatting."""
    model_str = ', '.join(MODEL_LIST)
    
    while True:
        model_name = input(f"Enter model name (default: gemini-2.0-flash): ").strip() or "gemini-2.0-flash"
        if model_name in MODEL_LIST:
            return model_name
        print(f"❌ Invalid model. Please choose from: {model_str}")

MODEL_NAME = get_user_model_input()
print(f"\n✅ Selected model: {MODEL_NAME}")
print(f"   Specs: {MODEL_SPECS[MODEL_NAME]}")

In [ ]:
# API Key Configuration (All Providers)
from getpass import getpass
try:
    from google.colab import userdata
    RUN_TIME = "colab"
except ImportError:
    RUN_TIME = "local"

def get_api_key(key_name, prompt):
    """Get API key from environment, Colab secrets, or user input."""
    # Check environment first
    if os.environ.get(key_name):
        print(f"  ✅ {key_name} (from environment)")
        return os.environ.get(key_name)
    
    # Try Colab secrets
    if RUN_TIME == "colab":
        try:
            key = userdata.get(key_name)
            if key:
                os.environ[key_name] = key
                print(f"  ✅ {key_name} (from Colab secrets)")
                return key
        except:
            pass
    
    # Prompt user
    key = getpass(f'{prompt}: ')
    if key:
        os.environ[key_name] = key
        print(f"  ✅ {key_name} set")
        return key
    return None

print("🔑 API Key Configuration")
print("   (Leave blank to skip, keys can also be set in Colab secrets)\n")

# Determine required keys based on model
if "gpt" in MODEL_NAME or "openai" in MODEL_NAME:
    get_api_key('OPENAI_API_KEY', 'Enter OPENAI_API_KEY')
elif "gemini" in MODEL_NAME:
    get_api_key('GEMINI_API_KEY', 'Enter GEMINI_API_KEY')
elif "claude" in MODEL_NAME or "anthropic" in MODEL_NAME:
    get_api_key('ANTHROPIC_API_KEY', 'Enter ANTHROPIC_API_KEY')
elif "deepseek" in MODEL_NAME:
    get_api_key('DEEPSEEK_API_KEY', 'Enter DEEPSEEK_API_KEY')
elif "llama" in MODEL_NAME or "qwen" in MODEL_NAME or "mistral" in MODEL_NAME:
    get_api_key('TOGETHERAI_API_KEY', 'Enter TOGETHERAI_API_KEY')

# Tool API keys (always needed)
print("\n🛠️ Tool API Keys:")
get_api_key('TAVILY_API_KEY', 'Enter TAVILY_API_KEY (for web search)')
get_api_key('NVD_API_KEY', 'Enter NVD_API_KEY (optional, for faster lookups)')

print("\n✅ API configuration complete!")

# Model Configuration

In [ ]:
# Configure DSPy Model
import dspy

def configure_dspy_model(model_name):
    """Configure DSPy with the specified model and API key."""
    # Determine provider and API key
    if "openai" in model_name or "gpt" in model_name:
        provider, key_var = "openai", "OPENAI_API_KEY"
    elif "gemini" in model_name:
        provider, key_var = "gemini", "GEMINI_API_KEY"
    elif "claude" in model_name or "anthropic" in model_name:
        provider, key_var = "anthropic", "ANTHROPIC_API_KEY"
    elif "deepseek" in model_name:
        provider, key_var = "deepseek", "DEEPSEEK_API_KEY"
    elif "llama" in model_name or "qwen" in model_name or "mistral" in model_name:
        provider, key_var = "together_ai", "TOGETHERAI_API_KEY"
    else:
        raise ValueError(f"Unknown model provider for: {model_name}")
    
    api_key = os.environ.get(key_var)
    if not api_key:
        raise ValueError(f"API key {key_var} not set!")
    
    lm = dspy.LM(
        model=f"{provider}/{model_name}",
        temperature=1.0,
        max_tokens=6000,
        api_key=api_key,
        cache=False
    )
    dspy.configure(lm=lm)
    return lm

# Configure the model
try:
    dspy_lm = configure_dspy_model(MODEL_NAME)
    print(f"✅ Successfully configured: {MODEL_NAME}")
except Exception as e:
    print(f"❌ Error configuring model: {e}")
    raise

# Run CVE Analysis (runner.py)

Execute the main vulnerability analysis pipeline.

In [ ]:
# Runner Parameters
MAX_CVES = 3  # Number of CVEs to analyze (adjust as needed)
SEED = 42     # Random seed for reproducibility
EVAL_MODE = False  # Set True to use evaluation dataset

print(f"📋 Runner Configuration:")
print(f"   Model: {MODEL_NAME}")
print(f"   Max CVEs: {MAX_CVES}")
print(f"   Seed: {SEED}")
print(f"   Eval Mode: {EVAL_MODE}")

In [ ]:
# Run CVE Analysis
print("🚀 Starting CVE Analysis Runner...")
print("=" * 70)

# Build command
cmd = f"python -m vul_analysis.src.runner --model {MODEL_NAME} --max-cves {MAX_CVES} --seed {SEED}"
if EVAL_MODE:
    cmd += " --eval"

# Execute
!{cmd}

print("=" * 70)
print("✅ Runner completed! Results saved to vul_analysis/logs/")

In [ ]:
# View Latest Results
import json
from pathlib import Path

log_dir = Path("vul_analysis/logs")
result_files = sorted(log_dir.glob("runner_test_results_*.json"), reverse=True)

if result_files:
    latest = result_files[0]
    print(f"📄 Latest Results: {latest.name}")
    print("=" * 60)
    
    with open(latest) as f:
        results = json.load(f)
    
    meta = results.get('metadata', {})
    preds = results.get('predictions', {})
    
    print(f"\n📊 Summary:")
    print(f"   Model: {meta.get('model', 'N/A')}")
    print(f"   CVEs Processed: {meta.get('total_cves_processed', 0)}")
    print(f"   Decision Hit Rate (DHR): {meta.get('DHR', 0):.2%}")
    print(f"   Total Time: {meta.get('total_time', 0):.1f}s")
    print(f"   Token Cost: ${meta.get('token_cost', 0):.4f}")
    
    print(f"\n📈 Predictions:")
    print(f"   TP/FP/TN/FN: {preds.get('TP/FP/TN/FN', 'N/A')}")
    print(f"   Precision: {preds.get('precision', 0):.2%}")
    print(f"   Recall: {preds.get('recall', 0):.2%}")
    print(f"   F1-Score: {preds.get('F1-Score', 0)}")
    print(f"   F2-Score: {preds.get('F2-Score', 0)}")
else:
    print("❌ No results found. Run the analysis first.")

---

# MIPROv2 Optimization (optimize_miprov2.py)

Optimize the CVE analysis agent using DSPy's MIPROv2 teleprompter for improved performance.

⚠️ **Note:** This is resource-intensive and may take significant time depending on dataset size.

In [ ]:
# MIPROv2 Parameters
MIPRO_MODEL = MODEL_NAME  # Use same model or change
AUTO_MODE = "light"       # Options: light, medium, heavy
TEMPERATURE = 1.0         # LLM temperature (0.5 - 1.5)
NUM_THREADS = 4           # Parallel threads
MIPRO_SEED = 42           # Random seed
USE_MINIBATCH = True      # Use minibatch training
MINIBATCH_SIZE = 5        # Minibatch size
GRID_SEARCH = False       # Enable hyperparameter grid search

print(f"📋 MIPROv2 Configuration:")
print(f"   Model: {MIPRO_MODEL}")
print(f"   Auto Mode: {AUTO_MODE}")
print(f"   Temperature: {TEMPERATURE}")
print(f"   Threads: {NUM_THREADS}")
print(f"   Minibatch: {USE_MINIBATCH} (size={MINIBATCH_SIZE})")
print(f"   Grid Search: {GRID_SEARCH}")

In [ ]:
# Run MIPROv2 Optimization
print("🧠 Starting MIPROv2 Optimization...")
print("⚠️  This may take a while depending on dataset size and model.")
print("=" * 70)

# Build command
cmd = f"""python -m vul_analysis.src.optimize_miprov2 \\
    --model {MIPRO_MODEL} \\
    --auto {AUTO_MODE} \\
    --temperature {TEMPERATURE} \\
    --num-threads {NUM_THREADS} \\
    --seed {MIPRO_SEED} \\
    --minibatch-size {MINIBATCH_SIZE}"""

if not USE_MINIBATCH:
    cmd += " --minibatch False"
if GRID_SEARCH:
    cmd += " --grid-search"

# Execute
!{cmd}

print("=" * 70)
print("✅ MIPROv2 optimization completed!")
print("   Optimized model saved to: vul_analysis/.cache/dspy_programs/")

---

# Utilities

In [ ]:
# List All Generated Files
from pathlib import Path

log_dir = Path("vul_analysis/logs")
cache_dir = Path("vul_analysis/.cache/dspy_programs")

print("📁 Log Files (latest 5):")
if log_dir.exists():
    for f in sorted(log_dir.glob("*.log"), reverse=True)[:5]:
        print(f"   📄 {f.name}")
else:
    print("   (no logs yet)")

print("\n📁 Result Files (latest 5):")
if log_dir.exists():
    for f in sorted(log_dir.glob("*.json"), reverse=True)[:5]:
        print(f"   📊 {f.name}")
else:
    print("   (no results yet)")

print("\n📁 Optimized Models:")
if cache_dir.exists():
    for d in sorted(cache_dir.glob("miprov2_*"), reverse=True)[:3]:
        print(f"   🧠 {d.name}")
else:
    print("   (no optimized models yet)")

In [ ]:
# Download Results (Colab)
try:
    from google.colab import files
    import shutil
    
    log_dir = Path("vul_analysis/logs")
    if log_dir.exists() and any(log_dir.iterdir()):
        shutil.make_archive("cve_analysis_results", 'zip', log_dir)
        files.download("cve_analysis_results.zip")
        print("✅ Download started!")
    else:
        print("❌ No results to download.")
except ImportError:
    print("⚠️ Not running in Colab. Results are in vul_analysis/logs/")